# ForecastClean — Chronos run (Colab / Kaggle)

Runs the Chronos foundation models on the same 9 series as the classical baselines.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower).  
**Status:** the adapter is unit-tested with stand-in models only — *this notebook is the first real test.* If a cell fails, copy the full error back to Claude.

The repo must be reachable by `git clone` (public repo, or use a token).

In [ ]:
# 1) get the code and install (chronos-forecasting pulls in torch/transformers if missing)
!git clone https://github.com/mirzayasirabdullahbaig07/ForecastClean.git
%cd ForecastClean
!pip -q install -e . chronos-forecasting
!pip show chronos-forecasting | head -3
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# 2) raw data is NOT in git (licences) -> download it here
!forecastclean download
!cat data/MANIFEST.csv | cut -d, -f1,5,7,8

In [ ]:
# 3) SMOKE TEST (a few minutes): one tiny model, two series, few windows
!forecastclean batch --only wiki_Python ecb_EURUSD --chronos amazon/chronos-bolt-tiny --max-origins 10 --out results/smoke

In [ ]:
# 4) FULL RUN: Bolt tiny/small + Chronos-2 on all series (CPU may take an hour or more)
!forecastclean batch --chronos amazon/chronos-bolt-tiny amazon/chronos-bolt-small amazon/chronos-2 --out results/chronos

In [ ]:
# 4b) THE MAIN ANALYSIS (gap A'): only windows AFTER the latest model release date + horizon sweep
!forecastclean batch --chronos amazon/chronos-bolt-small amazon/chronos-2 --release-cutoff --post-only --horizon-sweep --max-origins 50 --out results/chronos_post

In [ ]:
# 5) show + download results
import pandas as pd
s = pd.read_csv('results/chronos/summary.csv')
print(s.pivot_table(index='series', columns='model', values='rel_mase_geo').round(2).to_string())
!zip -qr chronos_results.zip results/chronos results/chronos_post data/MANIFEST.csv
from google.colab import files
files.download('chronos_results.zip')   # on Kaggle: the zip appears under /kaggle/working

## After the run
Save `chronos_results.zip`, unzip into your local `ForecastClean` folder and tell Claude. Do **not** commit `data/raw/` (licences). Record versions printed in step 1 in `EXPERIMENT_LOG.md`.